In [ ]:
from dotenv import load_dotenv
import os
import mysql.connector

load_dotenv()

True

In [21]:
host = os.environ.get('DB_URL')
user = os.environ.get('DB_USERNAME')
password = os.environ.get('DB_PASSWORD')

In [ ]:
DB_CONFIG = {
    'host': os.environ.get('DB_URL'),       # DB 서버 주소 (보통 로컬이면 'localhost')
    'user': os.environ.get('DB_USERNAME'), # MySQL 사용자 이름 (예: root)
    'password': os.environ.get('DB_PASSWORD'), # MySQL 사용자 비밀번호
    'port': 3307,  # Port 번호
    'database': 'assist',  # 데이터를 넣을 데이터베이스 이름
    'charset': 'utf8mb4'       # 한글 및 다양한 문자를 위한 인코딩 설정
}

DB connection open

In [ ]:
try:
    # 1. DB 연결
    mydb = mysql.connector.connect(**DB_CONFIG)

    # 2. 커서(Cursor) 생성: SQL 명령을 실행할 객체
    mycursor = mydb.cursor()
    # mycursor = mydb.cursor(dictionary=True) # 반환을 딕셔너리 형태로

    print("✅ MySQL 서버 연결 성공!")

except mysql.connector.Error as err:
    print(f"❌ MySQL 연결 오류: {err}")
    # 연결 실패 시 자세한 오류 메시지를 출력합니다.
    mydb = None
    mycursor = None

✅ MySQL 서버 연결 성공!


In [30]:
# db 연결 쿼리 실행 테스트

if 'mydb' in locals() and mydb and mydb.is_connected():
    try:
        mycursor.execute("SELECT DATABASE();") # 간단한 쿼리 예시
        result = mycursor.fetchone()
        print(f"현재 데이터베이스: {result[0]}")
        
        # 여기에 실제 원하는 쿼리 및 데이터 처리 로직을 넣습니다.
        # 예: mycursor.execute("SELECT * FROM ...")
        
    except mysql.connector.Error as err:
        print(f"❌ 쿼리 실행 오류: {err}")
else:
    print("⚠️ DB가 연결되지 않았습니다. DB connection을 먼저 실행해주세요.")

현재 데이터베이스: assist


기존 테이블 제거

In [ ]:
# 기존 테이블 삭제

if 'mydb' in locals() and mydb and mydb.is_connected():
    try:
        delete_dict_table_sql = """ 
        DROP TABLE tl_expert_dict;
        """
        mycursor.execute(delete_dict_table_sql)
        mydb.commit()
        
    except mysql.connector.Error as err:
        print(f"❌ 쿼리 실행 오류: {err}")
        mydb.rollback()
else:
    print("⚠️ DB가 연결되지 않았습니다. DB connection을 먼저 실행해주세요.")
    

테이블 생성

In [19]:
# DB 테이블 생성 sql문

if 'mydb' in locals() and mydb and mydb.is_connected():
    try:
        create_concept_table_sql = """
        -- 전문 용어 개념/설명 테이블 (설명서 역할)
        CREATE TABLE tl_dict_concept (
            concept_id BIGINT AUTO_INCREMENT NOT NULL COMMENT '개념 고유 ID',
            field VARCHAR(100) NOT NULL COMMENT '전문 분야 (e.g., IT, 금융, 문화)',
            description_ko TEXT NULL COMMENT '(기존 explanation) 한국어 설명',
            description_en TEXT NULL COMMENT '(신규) 영어 설명',
            description_ja TEXT NULL COMMENT '(신규/확장용) 일본어 설명',
            description_zh TEXT NULL COMMENT '(신규/확장용) 중국어 설명',
            created_at DATETIME NOT NULL COMMENT '생성 일시',
            updated_at DATETIME NOT NULL COMMENT '수정 일시',
            PRIMARY KEY (concept_id)
        ) COMMENT '전문 용어 개념/설명 테이블 (설명서 역할)';
        """
        mycursor.execute(create_concept_table_sql)
        

        create_term_table_sql = """ 
        -- 전문 용어 키워드 테이블 (단어/색인 역할)
        CREATE TABLE tl_dict_term (
            term_id BIGINT AUTO_INCREMENT NOT NULL COMMENT '용어 고유 ID',
            concept_id BIGINT NOT NULL COMMENT '(핵심) dict_concept의 ID를 참조',
            term_text VARCHAR(255) NOT NULL COMMENT '(기존 keyword) 실제 단어 (e.g., LiveKit)',
            language VARCHAR(10) NOT NULL COMMENT '(신규) 용어의 언어 (e.g., ko, en)',
            created_at DATETIME NOT NULL COMMENT '생성 일시',
            updated_at DATETIME NOT NULL COMMENT '수정 일시',
            PRIMARY KEY (term_id),
            FOREIGN KEY (concept_id) REFERENCES tl_dict_concept(concept_id) 
                ON UPDATE CASCADE  -- (선택적) concept_id가 변경되면 같이 변경
                ON DELETE RESTRICT -- (선택적) 개념에 연결된 용어가 있으면 개념 삭제 방지
        ) COMMENT '전문 용어 키워드 테이블 (단어/색인 역할)';
        """
        mycursor.execute(create_term_table_sql)

        mydb.commit()
        
    except mysql.connector.Error as err:
        print(f"❌ 쿼리 실행 오류: {err}")
        mydb.rollback()
else:
    print("⚠️ DB가 연결되지 않았습니다. DB connection을 먼저 실행해주세요.")
    

In [26]:
import pandas as pd

# CSV 파일을 DataFrame으로 읽어오기
try:
    file_path = './output/definitions.csv' # 👈 여기에 CSV 파일 경로를 지정하세요.
    definitions_df = pd.read_csv(file_path)
    print("✅ CSV 파일 로드 성공!")
    print(definitions_df.head()) # 상위 5개 행을 출력해 데이터 확인

    file_path = './output/terms.csv' # 👈 여기에 CSV 파일 경로를 지정하세요.
    terms_df = pd.read_csv(file_path)
    print("✅ CSV 파일 로드 성공!")
    print(terms_df.head()) # 상위 5개 행을 출력해 데이터 확인
    
except FileNotFoundError:
    print(f"❌ 오류: '{file_path}' 파일을 찾을 수 없습니다.")
except Exception as e:
    print(f"❌ CSV 로드 중 오류 발생: {e}")


✅ CSV 파일 로드 성공!
   def_id                                    English Meaning  \
0       1  A method of comparing two versions of a webpag...   
1       2  The content visible on a screen without scroll...   
2       3  Predefined requirements that must be met for a...   
3       4  The design of products and services to be usab...   
4       5  Data or insight that can be used to inform a d...   

                                      Korean Meaning  
0  두 가지 이상의 시안 중 어느 것이 더 나은 성과를 보이는지 사용자의 반응을 통해 ...  
1                 사용자가 스크롤을 내리지 않고 처음으로 보게 되는 화면 영역.  
2               개발된 기능이 요구사항을 충족했는지 판단하는 구체적인 조건 목록.  
3    장애 여부와 관계없이 누구나 동등하게 정보에 접근하고 이용할 수 있도록 보장하는 것.  
4                실제 행동이나 의사결정으로 이어질 수 있는, 구체적이고 유용한.  
✅ CSV 파일 로드 성공!
   def_id Language                 Word
0       1  English          A/B Testing
1       1   Korean              A/B 테스트
2       2   Korean          스크롤 없는 첫 화면
3       2  English       Above the Fold
4       3  English  Acceptance Criteria


In [31]:
if 'definitions_df' in locals() and 'mydb' in locals() and mydb.is_connected():
    try:
        sql = "INSERT INTO tl_dict_concept (field, description_en, description_ko, created_at, updated_at) VALUES ('IT', %s, %s, NOW(), NOW());"
        
        insert_count = 0

        for index, row in definitions_df.iterrows():
            
            values = (row['English Meaning'], row['Korean Meaning'])
            
            # 4. 쿼리 실행
            mycursor.execute(sql, values)
            insert_count += 1
            
        # 5. 모든 행 삽입 후, 트랜잭션 완료 (commit)
        mydb.commit()
        print(f"✅ {insert_count}개의 행이 성공적으로 삽입되었습니다.")
        
    except mysql.connector.Error as err:
        print(f"❌ 쿼리 실행 오류: {err}")
        mydb.rollback() # 오류 발생 시 롤백
    except KeyError as e:
        print(f"❌ DataFrame 컬럼명 오류: {e} - CSV 컬럼명을 확인하세요.")
else:
    print("⚠️ CSV 파일이 로드되지 않았거나 DB가 연결되지 않았습니다.")

✅ 603개의 행이 성공적으로 삽입되었습니다.


In [32]:
if 'terms_df' in locals() and 'mydb' in locals() and mydb.is_connected():
    try:
        sql = "INSERT INTO tl_dict_term (concept_id, term_text, language, created_at, updated_at) VALUES (%s, %s, %s, NOW(), NOW());"
        
        insert_count = 0

        for index, row in terms_df.iterrows():
            
            values = (row['def_id'], row['Word'], row['Language'])
            
            # 4. 쿼리 실행
            mycursor.execute(sql, values)
            insert_count += 1
            
        # 5. 모든 행 삽입 후, 트랜잭션 완료 (commit)
        mydb.commit()
        print(f"✅ {insert_count}개의 행이 성공적으로 삽입되었습니다.")
        
    except mysql.connector.Error as err:
        print(f"❌ 쿼리 실행 오류: {err}")
        mydb.rollback() # 오류 발생 시 롤백
    except KeyError as e:
        print(f"❌ DataFrame 컬럼명 오류: {e} - CSV 컬럼명을 확인하세요.")
else:
    print("⚠️ CSV 파일이 로드되지 않았거나 DB가 연결되지 않았습니다.")

✅ 1307개의 행이 성공적으로 삽입되었습니다.


In [ ]:
# 테이블 조회 확인(join 진행)

if 'mydb' in locals() and mydb and mydb.is_connected():
    try:
        sql = """
        select term_text, description_ko
        from tl_dict_term as term
        left join tl_dict_concept as concept
        on term.concept_id = concept.concept_id
        where term.term_id < 5;
        """
        mycursor.execute(sql)

        results = mycursor.fetchall()
        
        for (term, des) in results:
            print(f"단어: {term}, 한글 뜻: {des}")
        
    except mysql.connector.Error as err:
        print(f"❌ 쿼리 실행 오류: {err}")
else:
    print("⚠️ DB가 연결되지 않았습니다. DB connection을 먼저 실행해주세요.")

단어: A/B Testing, 한글 뜻: 두 가지 이상의 시안 중 어느 것이 더 나은 성과를 보이는지 사용자의 반응을 통해 비교하는 테스트.
단어: A/B 테스트, 한글 뜻: 두 가지 이상의 시안 중 어느 것이 더 나은 성과를 보이는지 사용자의 반응을 통해 비교하는 테스트.
단어: 스크롤 없는 첫 화면, 한글 뜻: 사용자가 스크롤을 내리지 않고 처음으로 보게 되는 화면 영역.
단어: Above the Fold, 한글 뜻: 사용자가 스크롤을 내리지 않고 처음으로 보게 되는 화면 영역.


DB connnection close

In [10]:
# 모든 작업이 끝났을 때 연결을 닫습니다.
if 'mydb' in locals() and mydb and mydb.is_connected():
    try:
        mycursor.close() # 커서 먼저 닫기
        mydb.close()
        print("✅ MySQL 연결이 성공적으로 종료되었습니다.")
    except mysql.connector.Error as err:
        print(f"❌ 연결 종료 오류: {err}")
else:
    print("ℹ️ 연결이 이미 닫혀있거나, 최초 연결에 실패했습니다.")

ℹ️ 연결이 이미 닫혀있거나, 최초 연결에 실패했습니다.
